## Purge Placeholder Author Records

DataCite metadata sometimes carries CDL/EZID **Kernel Metadata Reserved Codes** — sentinel values used when a `creators[].name` field is unavailable:

| Code | Meaning |
|---|---|
| `(:unav)` | value unavailable |
| `(:unkn)` | known to be unknown |
| `(:unap)` | not applicable |
| `(:unas)` | unassigned |
| `(:unal)` | unallowed/suppressed |
| `(:unac)` | temporarily inaccessible |

Some appear with literal trailing text (`(:Unkn) Unknown`, `(:Unas) Unassigned`, `(:unav) Unknown author`).

The DataCite forward fix (`notebooks/ingest/DataCite.py`, oxjob 143) prevents new placeholders from entering `datacite_parsed`. But existing residue persists in three places that aren't fed by `datacite_parsed` directly:

1. `openalex.works.work_authors` — flat author rows with `raw_author_name` matching the regex
2. `openalex.works.openalex_works_base.authorships[]` — the array elements still contain placeholder structs (this is the table feeding `work_authors` rebuilds, so it must be cleaned to stop the cleanup from getting reverted)
3. `openalex.authors.authors` — pure-placeholder profiles whose every `raw_author_names[]` entry is a placeholder (matching black-holes; e.g. A5133762328 with 1,509 works all attached via `(:unav)`)

This notebook **DELETEs** rows from #1 and #3, and **UPDATEs** #2 to filter placeholder elements out of the array. Snapshots are taken first; an audit log is built for rollback. **Mixed** profiles (real `full_name` + placeholder `display_name`) are left alone — `CreateAuthors` will re-derive `display_name` from the surviving real names.

**Forward fix is separate**: oxjob 145 will add the same regex filter to `notebooks/end2end/CreateWorksBase.ipynb` (the `exploded_authors` CTE) so this leak source can't recur from any provenance.

**Regex**: `^\s*\(:un[a-z]{2,3}\)(\s*(unknown( author)?|unassigned))?\s*$` (case-insensitive).

**Run order**: cells 1→6 are read-only snapshots and validations. Cells 7→9 are destructive — verify counts in the validation cells before running. Then re-run `notebooks/authors/CreateAuthors.ipynb` to rebuild `openalex_authors` without the deleted profiles and stale names.

### Cell 1: Snapshot work_authors candidates (rows to DELETE)

Captures every column for rollback insertion.

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.placeholder_purge_work_authors_snapshot AS
SELECT
  work_id, author_sequence, author_id, raw_author_name AS previous_raw_author_name,
  raw_affiliation_strings, is_corresponding,
  created_at AS previous_created_at, updated_at AS previous_updated_at
FROM openalex.works.work_authors
WHERE raw_author_name IS NOT NULL
  AND LOWER(raw_author_name) RLIKE '^\\s*\\(:un[a-z]{2,3}\\)(\\s*(unknown( author)?|unassigned))?\\s*$'

### Cell 2: work_authors validation

In [ ]:
SELECT
  COUNT(*) AS rows_to_delete,
  COUNT(DISTINCT author_id) AS distinct_authors_touched,
  COUNT(DISTINCT work_id) AS distinct_works_touched,
  SUM(CASE WHEN author_id IS NULL THEN 1 ELSE 0 END) AS rows_with_null_author_id
FROM openalex.authors.placeholder_purge_work_authors_snapshot

### Cell 3: Snapshot openalex_works_base candidates (works whose authorships array contains a placeholder element)

The full pre-update `authorships` array is captured for rollback.

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.placeholder_purge_works_base_snapshot AS
SELECT
  id AS work_id,
  authorships AS previous_authorships,
  current_timestamp() AS snapshot_timestamp
FROM openalex.works.openalex_works_base
WHERE EXISTS(authorships, a -> LOWER(a.raw_author_name) RLIKE '^\\s*\\(:un[a-z]{2,3}\\)(\\s*(unknown( author)?|unassigned))?\\s*$')

### Cell 4: openalex_works_base validation

`placeholder_elements_to_remove` counts the array entries that will be filtered out (NOT the number of works; one work may have multiple placeholder authorships).

In [ ]:
SELECT
  COUNT(*) AS works_to_update,
  SUM(SIZE(FILTER(previous_authorships, a -> LOWER(a.raw_author_name) RLIKE '^\\s*\\(:un[a-z]{2,3}\\)(\\s*(unknown( author)?|unassigned))?\\s*$'))) AS placeholder_elements_to_remove,
  SUM(SIZE(previous_authorships)) AS total_authorships_in_affected_works
FROM openalex.authors.placeholder_purge_works_base_snapshot

### Cell 5: Snapshot pure-placeholder profile candidates

Profiles in `openalex_authors` whose every `raw_author_names[]` entry is a placeholder. After cells 7 and 8 delete the underlying rows, these profiles will have empty raw_author_names; this snapshot captures them BEFORE deletion since the criterion depends on the current state.

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.placeholder_purge_profiles_snapshot AS
WITH per_author AS (
  SELECT
    oa.id, oa.display_name, oa.full_name, oa.orcid, oa.works_count, oa.raw_author_names,
    SIZE(oa.raw_author_names) AS total_names,
    SIZE(FILTER(oa.raw_author_names, n -> LOWER(n) RLIKE '^\\s*\\(:un[a-z]{2,3}\\)(\\s*(unknown( author)?|unassigned))?\\s*$')) AS placeholder_names
  FROM openalex.authors.openalex_authors oa
)
SELECT
  pa.id AS author_id,
  pa.display_name AS previous_display_name,
  pa.full_name AS previous_full_name,
  pa.orcid AS previous_orcid,
  pa.works_count AS previous_works_count,
  pa.raw_author_names AS previous_raw_author_names,
  -- Capture full source row from openalex.authors.authors for rollback
  TO_JSON(STRUCT(a.*)) AS previous_authors_row_json
FROM per_author pa
JOIN openalex.authors.authors a ON a.id = pa.id
WHERE pa.total_names > 0 AND pa.total_names = pa.placeholder_names

### Cell 6: profile candidates validation

In [ ]:
SELECT
  COUNT(*) AS profiles_to_delete,
  SUM(CASE WHEN previous_orcid IS NOT NULL AND previous_orcid <> '' THEN 1 ELSE 0 END) AS profiles_with_orcid,
  SUM(previous_works_count) AS total_works_currently_attached
FROM openalex.authors.placeholder_purge_profiles_snapshot

### Cell 7: Combined audit log (for rollback)

Records what's about to be deleted/updated. Three scopes: `work_author_row` (DELETEd row's prior values), `works_base_authorships` (pre-update authorships array as JSON), `author_profile` (pre-delete authors row as JSON).

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.placeholder_purge_log AS
SELECT 'work_author_row' AS scope,
  CAST(work_id AS STRING) AS key1,
  CAST(author_sequence AS STRING) AS key2,
  CAST(author_id AS STRING) AS key3,
  previous_raw_author_name AS previous_value_text,
  CAST(NULL AS STRING) AS previous_value_json,
  current_timestamp() AS purge_timestamp
FROM openalex.authors.placeholder_purge_work_authors_snapshot
UNION ALL
SELECT 'works_base_authorships' AS scope,
  CAST(work_id AS STRING) AS key1,
  CAST(NULL AS STRING) AS key2,
  CAST(NULL AS STRING) AS key3,
  CAST(NULL AS STRING) AS previous_value_text,
  TO_JSON(previous_authorships) AS previous_value_json,
  current_timestamp() AS purge_timestamp
FROM openalex.authors.placeholder_purge_works_base_snapshot
UNION ALL
SELECT 'author_profile' AS scope,
  CAST(author_id AS STRING) AS key1,
  CAST(NULL AS STRING) AS key2,
  CAST(NULL AS STRING) AS key3,
  previous_full_name AS previous_value_text,
  previous_authors_row_json AS previous_value_json,
  current_timestamp() AS purge_timestamp
FROM openalex.authors.placeholder_purge_profiles_snapshot

### Cell 8: EXECUTE — DELETE placeholder rows from work_authors

**WARNING**: Destructive. Verify cell 2 row count before running. The MERGE is gated on the prior `raw_author_name` matching the snapshot to avoid clobbering anything that changed since cell 1.

In [ ]:
MERGE INTO openalex.works.work_authors AS target
USING openalex.authors.placeholder_purge_work_authors_snapshot AS source
  ON target.work_id = source.work_id
     AND target.author_sequence = source.author_sequence
WHEN MATCHED AND target.raw_author_name = source.previous_raw_author_name THEN DELETE

### Cell 9: EXECUTE — UPDATE openalex_works_base.authorships to filter out placeholder elements

**WARNING**: Destructive. Verify cell 4 counts before running. This is the table that feeds `work_authors` rebuilds — without this, end2end will re-introduce placeholder rows.

In [ ]:
MERGE INTO openalex.works.openalex_works_base AS target
USING openalex.authors.placeholder_purge_works_base_snapshot AS source
  ON target.id = source.work_id
WHEN MATCHED THEN
  UPDATE SET target.authorships =
    FILTER(target.authorships, a -> NOT (LOWER(a.raw_author_name) RLIKE '^\\s*\\(:un[a-z]{2,3}\\)(\\s*(unknown( author)?|unassigned))?\\s*$'))

### Cell 10: EXECUTE — DELETE pure-placeholder profiles from openalex.authors.authors

**WARNING**: Destructive. Verify cell 6 counts before running. Only profiles where every `raw_author_names[]` entry is a placeholder are deleted; mixed profiles stay (CreateAuthors will re-derive their display_name on rebuild).

In [ ]:
MERGE INTO openalex.authors.authors AS target
USING openalex.authors.placeholder_purge_profiles_snapshot AS source
  ON target.id = source.author_id
WHEN MATCHED THEN DELETE

### Cell 11: Post-cleanup verification

All three counts should be 0. Audit log row count should equal the sum of pre-purge counts.

In [ ]:
SELECT 'work_authors_remaining' AS check_name, COUNT(*) AS remaining
FROM openalex.works.work_authors
WHERE raw_author_name IS NOT NULL
  AND LOWER(raw_author_name) RLIKE '^\\s*\\(:un[a-z]{2,3}\\)(\\s*(unknown( author)?|unassigned))?\\s*$'
UNION ALL
SELECT 'works_base_authorships_remaining' AS check_name, COUNT(*) AS remaining
FROM openalex.works.openalex_works_base
WHERE EXISTS(authorships, a -> LOWER(a.raw_author_name) RLIKE '^\\s*\\(:un[a-z]{2,3}\\)(\\s*(unknown( author)?|unassigned))?\\s*$')
UNION ALL
SELECT 'authors_full_name_placeholder_remaining' AS check_name, COUNT(*) AS remaining
FROM openalex.authors.authors
WHERE full_name IS NOT NULL
  AND LOWER(full_name) RLIKE '^\\s*\\(:un[a-z]{2,3}\\)(\\s*(unknown( author)?|unassigned))?\\s*$'
UNION ALL
SELECT 'pure_placeholder_profiles_remaining' AS check_name, COUNT(*) AS remaining
FROM (
  SELECT id,
         SIZE(raw_author_names) AS total_names,
         SIZE(FILTER(raw_author_names, n -> LOWER(n) RLIKE '^\\s*\\(:un[a-z]{2,3}\\)(\\s*(unknown( author)?|unassigned))?\\s*$')) AS placeholder_names
  FROM openalex.authors.openalex_authors
  WHERE raw_author_names IS NOT NULL AND SIZE(raw_author_names) > 0
)
WHERE total_names = placeholder_names AND total_names > 0
UNION ALL
SELECT 'audit_log_rows' AS check_name, COUNT(*) AS remaining
FROM openalex.authors.placeholder_purge_log

### Cell 12: Rollback templates (use only if a systematic FP is discovered)

Run any subset; each scope is independent.

```sql
-- 1) Restore work_authors rows (INSERT back from snapshot)
INSERT INTO openalex.works.work_authors
  (work_id, author_sequence, author_id, raw_author_name, raw_affiliation_strings,
   is_corresponding, created_at, updated_at)
SELECT work_id, author_sequence, author_id, previous_raw_author_name,
       raw_affiliation_strings, is_corresponding,
       previous_created_at, current_timestamp()
FROM openalex.authors.placeholder_purge_work_authors_snapshot;

-- 2) Restore openalex_works_base.authorships (overwrite filtered array with snapshot)
MERGE INTO openalex.works.openalex_works_base AS target
USING openalex.authors.placeholder_purge_works_base_snapshot AS source
  ON target.id = source.work_id
WHEN MATCHED THEN UPDATE SET target.authorships = source.previous_authorships;

-- 3) Restore deleted author profiles (parse the captured JSON; manual mapping required
--    because openalex.authors.authors has many columns — extract from previous_value_json
--    in placeholder_purge_log and INSERT row-by-row).
```

Add a `WHERE` clause inside the source subquery to scope rollback to a specific class.

### Downstream rebuild

After cells 8–10, kick off `notebooks/authors/CreateAuthors.ipynb` so:
- `openalex_authors.raw_author_names` drops the now-deleted placeholder rows
- The 250 deleted profiles disappear from `openalex_authors`
- For mixed profiles (kept here), `display_name` re-derives from the remaining real raw_author_names — should self-heal placeholder display_names for profiles like A5125162398 ('Leuenberger, Heinz').

ES sync sees `updated_date` bumps on touched works (via the `openalex_works_base` UPDATE) and on profiles via the `CreateAuthors` rebuild.